# 11a. 롤링 추론 기반 하이퍼파라미터 최적화 (Optuna)

이 노트북은 **디스크 단위의 롤링 알람 평가 메트릭**을 직접 목적 함수(Objective Function)로 사용하여 LightGBM의 하이퍼파라미터를 최적화합니다.

---

## 📋 데이터 파이프라인 가이드

### 1. 입력 데이터 (Inputs)
- **학습용 언더배깅 서브셋 (Subsets)**:
  - 경로: `config.train_config.SUBSET_DIR` (`data/06b_subset_generation/seed_42/subset_*.parquet`)
  - 설명: 비대칭 샘플링을 적용해 정상:고장 비율이 10:1로 조절된 10개의 학습 서브셋 데이터프레임입니다.
- **검증 데이터셋 (Validation)**:
  - 경로: `config.train_config.VAL_TUNE_PATH` (`data/split_group_stratified/val_tune.parquet`)
  - 설명: 하이퍼파라미터 및 임계값 튜닝용 전체 검증셋 데이터프레임입니다. (행 샘플링 없이 사용)
- **사용 피처 목록 (Features)**:
  - 경로: `config.train_config.FEATURE_COLS` (27개 파생 변수 리스트)

### 2. 주요 수행 작업 (Process)
1. **검증 데이터 인덱스 사전 매핑 (Pre-Structuring)**:
   - 검증 데이터셋(`val_tune`)을 `base_serial`로 그룹화하고 시간 순서(`date`)로 정렬합니다.
   - 각 물리 디스크의 예측 결과가 저장될 행(Row) 인덱스 및 고장 시점을 1회만 미리 매핑하여 루프 내 연산 속도를 극대화합니다.
2. **하이퍼파라미터 탐색 (Optuna Trials)**:
   - `max_depth`, `num_leaves`, `learning_rate`, `min_child_samples`, `feature_fraction`, `n_estimators`를 변경하며 모델을 학습합니다.
3. **앙상블 학습 및 추론 (Soft-voting Ensemble)**:
   - 제안된 파라미터 조합으로 10개의 서브셋 학습기를 개별 학습한 뒤, 정렬된 검증 데이터셋 전체에 대한 소프트보팅 예측 확률을 연산합니다.
4. **임계값 스캔 및 디스크 단위 평가 (Rolling Alarm Objective)**:
   - `0.5` ~ `0.999` 범위의 임계값에 대해 롤링 알람 조건(14일 슬라이딩 윈도우 내 2회 이상 알람 발생)을 모사합니다.
   - **`FAR(오탐율) <= 1.0%` 제약을 만족**하는 임계값 구간을 스캔하고, 해당 조건에서 달성한 **최대 `Recall_30d` (30일 전 탐지율) 점수**를 최적화 목표(maximize)로 Optuna에 반환합니다.

### 3. 최종 산출 결과물 및 저장 방식 (Outputs & Saving)
탐색 완료 후 가장 높은 롤링 Recall 점수를 기록한 최적 모델 패키지는 다음 디렉토리에 영구 저장됩니다.
- **최종 저장 디렉토리**: `models/underbagging_ensemble_rolling_opt/` (자동 생성)
- **저장 파일 일람**:
  1. `subset_00.pkl` ~ `subset_09.pkl`: 최종 최적 앙상블 LightGBM 모델 객체들 (10개)
  2. `best_params.json`: 최적 하이퍼파라미터 설정 정보
  3. `best_threshold.json`: 오탐율 1.0%를 제약으로 만족하는 최적 임계값(`threshold`), 최소 알람 발생 횟수(`min_alarms=2`), 윈도우 크기(`window_size=14`) 및 기록된 검증 Recall 점수 메타데이터
  4. `feature_cols.json`: 최종 모델 학습에 사용된 27개 피처 명단 리스트

---


In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.abspath('../..'))  # notebooks/11_rolling_optuna/ 에서 상위 두 수준

import numpy as np
import pandas as pd
import time
import optuna
import json
import joblib
from pathlib import Path

from src.train_core import (
    SubsetTrainer, UnderbaggingEnsemble,
    print_ensemble_summary, plot_subset_prauc, plot_confusion_matrix
)
import config.train_config as cfg

print('✅ 환경 준비 완료')
print(f'TRAIN_PATH    : {cfg.TRAIN_PATH}')
print(f'VAL_TUNE_PATH : {cfg.VAL_TUNE_PATH}')


c:\Workspace\06_ML_projdect\26_1_COIN\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 환경 준비 완료
TRAIN_PATH    : C:\Workspace\06_ML_projdect\26_1_COIN\data\split_group_stratified\train.parquet
VAL_TUNE_PATH : C:\Workspace\06_ML_projdect\26_1_COIN\data\split_group_stratified\val_tune.parquet


## 1. 데이터 로드 및 롤링 평가용 구조 사전 준비

속도 최적화를 위해 검증 셋(`val_tune`)을 `base_serial` 기준으로 정렬 및 그룹화하여 각 디스크의 예측 대상 행 인덱스를 미리 매핑해둡니다.

In [2]:
# 데이터셋 로드
df_train = pd.read_parquet(cfg.TRAIN_PATH)
df_val_tune = pd.read_parquet(cfg.VAL_TUNE_PATH)

# 사용할 피처 설정
_meta = {'serial_number', 'date', 'failure', cfg.TARGET_COL}
FEATURE_COLS = cfg.FEATURE_COLS or [c for c in df_train.columns if c not in _meta]

print(f'train    : {len(df_train):,} rows')
print(f'val_tune : {len(df_val_tune):,} rows')
print(f'features : {len(FEATURE_COLS)} 개')

print("\n⚡ [초고속 롤링] 검증 데이터셋 사전 인덱스 매핑 중...")
df_val_tune_s = df_val_tune.copy()
df_val_tune_s["base_serial"] = df_val_tune_s["serial_number"].str.replace(r'_\d+$', '', regex=True)
df_val_tune_s = df_val_tune_s.sort_values(["base_serial", "date"]).reset_index(drop=True)

disks_structure = []
for base_serial, grp in df_val_tune_s.groupby("base_serial"):
    is_failed = int(grp[cfg.TARGET_COL].max())
    fail_date = None
    early_mask = None
    valid_mask = None
    if is_failed:
        fail_date = pd.Timestamp(grp.loc[grp[cfg.TARGET_COL] == 1, "date"].max())
        leads = (fail_date - pd.to_datetime(grp["date"])).dt.days.values
        early_mask = (leads > 30)
        valid_mask = (leads >= 0) & (leads <= 30)
        
    disks_structure.append({
        "base_serial": base_serial,
        "is_failed": is_failed,
        "dates": grp["date"].values,
        "fail_date": fail_date,
        "indices": grp.index.values, # 전체 검증 정렬 데이터프레임 내에서의 row indices
        "early_mask": early_mask,
        "valid_mask": valid_mask
    })

print(f"✅ 준비 완료: 총 {len(disks_structure):,}개 물리 디스크 사전 매핑 완료.")

train    : 47,252,074 rows
val_tune : 7,902,193 rows
features : 27 개

⚡ [초고속 롤링] 검증 데이터셋 사전 인덱스 매핑 중...
✅ 준비 완료: 총 3,686개 물리 디스크 사전 매핑 완료.


## 2. 롤링 추론 목적 함수 정의

앙상블 모델 학습 후, 0.5부터 0.999까지의 임계값 그리드를 고속 스캔하여 `FAR <= 1.0%` 제약을 만족하는 최적 임계값에서의 Recall을 점수로 반환합니다.

In [3]:
def make_rolling_optuna_objective(
    df_val_sorted: pd.DataFrame,
    disks_struct: list[dict],
    feature_cols: list[str],
    target_col: str = "failure",
    target_far: float = 1.0,   # 목표 FAR 제한 (%)
    lead_time: int = 30,       # 목표 Lead-time (일)
    device: str = "cpu",
    bounds: dict = None,
):
    # 훈련 서브셋 로드
    subset_dir = Path(cfg.SUBSET_DIR)
    subset_files = sorted(list(subset_dir.glob("subset_*.parquet")))
    train_subsets = [pd.read_parquet(f) for f in subset_files]

    # [NumPy 기반 고속 롤링 2번째 최댓값 필터]
    def get_rolling_2nd_largest(probs, window=14):
        L = len(probs)
        padded = np.empty(L + window - 1, dtype=probs.dtype)
        padded[:window - 1] = -1e9  # 패딩 영역은 아주 작은 값으로 처리하여 무영향화
        padded[window - 1:] = probs
        v = np.lib.stride_tricks.sliding_window_view(padded, window)
        return np.partition(v, -2, axis=1)[:, -2]

    def objective(trial):
        # 하이퍼파라미터 탐색 공간 정의
        max_depth    = trial.suggest_int("max_depth", *bounds["max_depth"])
        max_leaves   = min(2 ** max_depth, bounds["num_leaves"][1]) if "num_leaves" in bounds else 64
        num_leaves   = trial.suggest_int("num_leaves", min(16, max_leaves), max_leaves)
        n_estimators = trial.suggest_int("n_estimators", *bounds["n_estimators"])
        
        params = {
            "learning_rate":     trial.suggest_float("learning_rate", *bounds["learning_rate"], log=True),
            "max_depth":         max_depth,
            "num_leaves":        num_leaves,
            "min_child_samples": trial.suggest_int("min_child_samples", *bounds["min_child_samples"]),
            "feature_fraction":  trial.suggest_float("feature_fraction", *bounds["feature_fraction"]),
            "bagging_fraction":  trial.suggest_float("bagging_fraction", *bounds["bagging_fraction"]),
            "bagging_freq":      1, # 배깅을 활성화하기 위해 1로 고정
            "lambda_l1":         trial.suggest_float("lambda_l1", *bounds["lambda_l1"], log=True),
            "lambda_l2":         trial.suggest_float("lambda_l2", *bounds["lambda_l2"], log=True),
            "verbosity":         -1,
            "device":            device,
            "random_state":      42,
            "n_estimators":      n_estimators,
            "max_bin":           63,
        }

        # 앙상블 학습 시 trial을 명시적으로 넘겨주어 현재 트라이얼 번호가 출력되도록 함
        trainer = SubsetTrainer(lgbm_params=params, target_col=target_col)
        ens     = UnderbaggingEnsemble(trainer=trainer)
        result  = ens.fit(train_subsets, df_val_sorted, feature_cols=feature_cols, target_col=target_col, trial=trial)

        # [개선 1] 중복 추론 제거: fit 내에서 연산된 val_tune_probs를 바로 참조하여 시간 절약
        y_prob = result.val_tune_probs

        # [개선 2] NumPy 벡터화를 위한 디스크별 임계 조건 벡터 연산
        T_normal = []
        T_early = []
        T_valid = []

        for d in disks_struct:
            disk_probs = y_prob[d["indices"]]
            # sliding window = 14일 내 최소 2회 알람 조건 결정자 (M[i] >= T 이면 롤링 2회 알람 참)
            M = get_rolling_2nd_largest(disk_probs, window=14)
            
            if not d["is_failed"]:
                T_normal.append(M.max())
            else:
                m_early = M[d["early_mask"]]
                T_early.append(m_early.max() if len(m_early) > 0 else -1.0)
                
                m_valid = M[d["valid_mask"]]
                T_valid.append(m_valid.max() if len(m_valid) > 0 else -1.0)

        T_normal = np.array(T_normal)
        T_early = np.array(T_early)
        T_valid = np.array(T_valid)

        best_recall = 0.0
        best_thr = 0.5
        
        # 임계값 그리드 고속 스캔 (NumPy 벡터화 비교 연산)
        for threshold in np.linspace(0.5, 0.999, 100):
            far = np.mean(T_normal >= threshold) * 100
            
            if far <= target_far:
                hits = (threshold > T_early) & (threshold <= T_valid)
                recall = np.mean(hits) * 100
                if recall > best_recall:
                    best_recall = recall
                    best_thr = threshold

        # 임계값 기록 저장
        trial.set_user_attr("best_threshold", best_thr)
        return best_recall

    return objective

## 3. Optuna 최적화 실행

목표 FAR 제약(FPR <= 1.0%) 내에서 30일 고장 탐지율(Recall)을 최대화하는 하이퍼파라미터를 탐색합니다.

In [ ]:
bounds = cfg.OPTUNA_BOUNDS.copy()

# 추가 하이퍼파라미터의 탐색 범위 정의
if "bagging_fraction" not in bounds:
    bounds["bagging_fraction"] = (0.6, 1.0)
if "lambda_l1" not in bounds:
    bounds["lambda_l1"] = (1e-8, 10.0)
if "lambda_l2" not in bounds:
    bounds["lambda_l2"] = (1e-8, 10.0)

optuna_temp_dir = Path(cfg.MODEL_SAVE_DIR).parent / "optuna_temp_rolling"
optuna_temp_dir.mkdir(parents=True, exist_ok=True)

db_path = getattr(cfg, "OPTUNA_DB_PATH", "optuna_study.db")
study_name = "hdd_failure_rolling_opt"
storage_url = f"sqlite:///{db_path}"

study = optuna.create_study(
    study_name=study_name,
    storage=storage_url,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=cfg.SEED),
    load_if_exists=True,
)

# ── [추가] 중단되어 RUNNING 상태로 방치된 유령 트라이얼 정리 ──
for t in study.trials:
    if t.state == optuna.trial.TrialState.RUNNING:
        try:
            study.storage.set_trial_state(t._trial_id, optuna.trial.TrialState.FAIL)
        except Exception:
            pass

obj = make_rolling_optuna_objective(
    df_val_sorted=df_val_tune_s,
    disks_struct=disks_structure,
    feature_cols=FEATURE_COLS,
    target_col=cfg.TARGET_COL,
    target_far=1.0,  # 롤링 FAR 1.0% 이하
    lead_time=30,    # 30일 리드타임
    device=cfg.LGBM_PARAMS.get("device", "cpu"),
    bounds=bounds
)

# 목표 완료 횟수 (COMPLETE) 설정
target_complete_trials = 100
completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
n_remaining = target_complete_trials - len(completed_trials)

if n_remaining > 0:
    print(f"🚀 롤링 메트릭 기반 Optuna 최적화 시작 (목표: {target_complete_trials}회 COMPLETE)")
    print(f"   - 현재 완료된 트라이얼: {len(completed_trials)}회")
    print(f"   - 추가로 실행할 트라이얼: {n_remaining}회")
    study.optimize(obj, n_trials=n_remaining)
else:
    print(f"✅ 이미 목표 완료 횟수({target_complete_trials}회)를 달성하였습니다. 추가 실행하지 않습니다.")

best_trial = study.best_trial
print('\n=========================================')
print(f'🏆 최적 트라이얼: Trial {best_trial.number}')
print(f'  - 최적 Recall_30d: {best_trial.value:.4f}%')
print(f'  - 최적 임계값(T): {best_trial.user_attrs.get("best_threshold"):.4f}')
print('=========================================')

[I 2026-05-27 20:29:25,554] Using an existing study with name 'hdd_failure_rolling_opt' instead of creating a new one.


🚀 롤링 메트릭 기반 Optuna 최적화 시작 (목표: 100회 COMPLETE)
   - 현재 완료된 트라이얼: 0회
   - 추가로 실행할 트라이얼: 100회


[I 2026-05-27 20:37:03,630] Trial 3 finished with value: 23.392857142857142 and parameters: {'max_depth': 6, 'num_leaves': 62, 'n_estimators': 693, 'learning_rate': 0.03968793330444373, 'min_child_samples': 32, 'feature_fraction': 0.662397808134481, 'bagging_fraction': 0.6232334448672797, 'lambda_l1': 0.6245760287469893, 'lambda_l2': 0.002570603566117598}. Best is trial 3 with value: 23.392857142857142.


[I 2026-05-27 20:41:12,363] Trial 4 finished with value: 22.857142857142858 and parameters: {'max_depth': 8, 'num_leaves': 18, 'n_estimators': 788, 'learning_rate': 0.06798962421591129, 'min_child_samples': 37, 'feature_fraction': 0.6727299868828402, 'bagging_fraction': 0.6733618039413735, 'lambda_l1': 5.472429642032198e-06, 'lambda_l2': 0.00052821153945323}. Best is trial 3 with value: 23.392857142857142.


[I 2026-05-27 20:48:10,869] Trial 5 finished with value: 23.214285714285715 and parameters: {'max_depth': 7, 'num_leaves': 48, 'n_estimators': 645, 'learning_rate': 0.013787764619353767, 'min_child_samples': 43, 'feature_fraction': 0.7465447373174767, 'bagging_fraction': 0.7824279936868144, 'lambda_l1': 0.1165691561324743, 'lambda_l2': 6.267062696005991e-07}. Best is trial 3 with value: 23.392857142857142.


...

  🏋️  Trial 30 - Subset 1/10 학습 중...                        

[I 2026-05-27 21:57:49,638] Trial 30 pruned. 


  🚫  [Pruned] Trial 30 pruned at step 0 (score: 0.11584)   
  🏋️  Trial 31 - Subset 1/10 학습 중...                        

[I 2026-05-27 21:58:24,755] Trial 31 pruned. 


  🚫  [Pruned] Trial 31 pruned at step 0 (score: 0.11399)   
  🏋️  Trial 32 - Subset 1/10 학습 중...                        

## 4. 최종 모델 저장 및 배포

최적의 롤링 Recall을 보여준 앙상블 모델들과 설정 파일들을 지정된 모델 경로에 영구 저장합니다.

In [ ]:
import json
import shutil
import joblib

# 1. 최적 파라미터 로드 및 결합
best_params = cfg.LGBM_PARAMS.copy()
best_params.update(best_trial.params)

# 2. 롤링 최적화 전용 모델 폴더 생성
final_model_dir = Path(cfg.MODEL_SAVE_DIR).parent / "underbagging_ensemble_rolling_opt"
final_model_dir.mkdir(parents=True, exist_ok=True)

# 3. 최적 파라미터 기반 deterministic 최종 모델 재학습 및 파일명 규칙(subset_*.pkl) 적용 저장
print(f"💾 최적 모델 최종 재학습 및 영구 저장 중: {final_model_dir}")
trainer = SubsetTrainer(lgbm_params=best_params, target_col=cfg.TARGET_COL)
ens     = UnderbaggingEnsemble(trainer=trainer)

# train subsets 로드
subset_dir = Path(cfg.SUBSET_DIR)
subset_files = sorted(list(subset_dir.glob("subset_*.parquet")))
df_train = [pd.read_parquet(f) for f in subset_files]

result = ens.fit(df_train, df_val_tune_s, feature_cols=FEATURE_COLS, target_col=cfg.TARGET_COL)

for idx_m, model in enumerate(result.models):
    # 명명 규칙 불일치(다운스트림 로드 에러) 버그 해결
    model_name = f"subset_{idx_m:02d}.pkl"
    joblib.dump(model, final_model_dir / model_name)
    print(f"  → 저장 완료: {model_name}")

# 파라미터 및 하이퍼파라미터 JSON 메타데이터 저장
with open(final_model_dir / "best_params.json", "w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2, ensure_ascii=False)

with open(final_model_dir / "best_threshold.json", "w", encoding="utf-8") as f:
    json.dump({
        "threshold": best_trial.user_attrs.get("best_threshold"),
        "min_alarms": 2,
        "window_size": 14,
        "val_rolling_recall": best_trial.value,
        "target_far": 1.0
    }, f, indent=2, ensure_ascii=False)

with open(final_model_dir / "feature_cols.json", "w", encoding="utf-8") as f:
    json.dump(FEATURE_COLS, f, indent=2, ensure_ascii=False)

# 임시 모델 디렉토리 정리 (최적화 중 생성되지 않았더라도 방어적으로 수행)
shutil.rmtree(optuna_temp_dir, ignore_errors=True)

print("\n🎉 모든 최적화 완료 및 최종 모델 패키징 완료!")